In [1]:
from data.datafinder import DataFinder

import pandas as pd
from functools import partial
import json
import random
from dataclasses import asdict
from pathlib import Path

from dataset_recsys.ingestion.fetch_gems_datasets import DatasetProfile
from dataset_recsys.embeddings import build_embedding_text, build_raw_embedding_text, encode_texts
from dataset_recsys.retrieval import build_recommendations
from dataset_recsys.utils.bedrock import enrich_batch
from dataset_recsys.utils.text_preprocessing import preprocess_profiles_field
from dataset_recsys.workflows.full_batch_rebuild import FullBatchRebuildWorkflow

from recs_metrics.item_item import recall_at_n, tndcg_at_n

EVAL_CUTOFFS = [10, 20, 50]
CONNECTED_QUERY_COUNT = 50
ADDITIONAL_DATASET_COUNT = 100
RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

DEMO_OUTPUT_DIR = Path("data/datafinder/demo_outputs")
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ENRICHMENT_LLM = "claude-sonnet-4-6"
PROMPT_VERSION = "catalog_summary_v1"
EMBEDDING_MODEL = "allenai/specter2_base"

In [2]:
df = DataFinder()
data = df.get()
corpus = data["corpus"]
ground_truth_links = df.get_links_from_queries()

##### We select a fixed number of query datasets, add all their linked neighbors, and then add a small number of extra distractor datasets to make the problem more realistic without running Bedrock on the full corpus.

In [3]:
all_query_ids = sorted(ground_truth_links.keys())
selected_query_ids = rng.sample(all_query_ids, min(CONNECTED_QUERY_COUNT, len(all_query_ids)))
selected_dataset_ids = set(selected_query_ids)
for query_id in selected_query_ids:
    selected_dataset_ids.update(ground_truth_links.get(query_id, []))

remaining_dataset_ids = sorted(set(corpus["id"]) - selected_dataset_ids)
extra_ids = rng.sample(remaining_dataset_ids, min(ADDITIONAL_DATASET_COUNT, len(remaining_dataset_ids)))
selected_dataset_ids.update(extra_ids)

corpus = corpus[corpus["id"].isin(selected_dataset_ids)].copy()

selected_query_id_set = set(selected_query_ids)
ground_truth_links = {
    query_id: {target_id for target_id in targets if target_id in selected_dataset_ids}
    for query_id, targets in ground_truth_links.items()
    if query_id in selected_query_id_set
}

##### We transform heterogeneous Datafinder metadata into the unified DatasetProfile format used by the recommender pipeline.

In [4]:
def datafinder_to_dataset_profiles(row: pd.Series) -> DatasetProfile:
    title = row["title"] if pd.notna(row["title"]) else str(row["id"])
    description = row["description"] if pd.notna(row["description"]) else ""

    keyword: list[str] = []
    if isinstance(row["tasks"], list):
        keyword.extend(str(task) for task in row["tasks"] if task)

    return DatasetProfile(
        id=str(row["id"]),
        title=str(title),
        headline="",
        description=str(description),
        keywords=", ".join(keyword),
        field_of_science="",
        catalog_summary="", # to generate with LLM
    )

profiles = corpus.apply(datafinder_to_dataset_profiles, axis=1).tolist()
profiles = preprocess_profiles_field(profiles, field_name="description")

In [5]:
print(f"Total datasets sampled: {len(profiles)}")

Total datasets sampled: 397


##### Run the full batch workflow (demo mode)
##### We reuse the same workflow orchestration as the portal recommender, but without storing results DBs.

In [ ]:
workflow_outputs: dict[str, object] = {
    "dataset_profiles": None,
    "recommendations": None,
}


def fetch_catalog_step() -> list[DatasetProfile]:
    return profiles

def preprocess_catalog_step(catalog: list[DatasetProfile]) -> list[DatasetProfile]:
    processed_catalog = preprocess_profiles_field(catalog, field_name="catalog_summary")
    workflow_outputs["dataset_profiles"] = processed_catalog
    return processed_catalog

def generate_embeddings_step(catalog: list[DatasetProfile]):
    embedding_texts = [build_embedding_text(profile) for profile in catalog]
    return encode_texts(embedding_texts, model_name=EMBEDDING_MODEL)

def compute_recommendations_step(embeddings, catalog: list[DatasetProfile]):
    return build_recommendations(catalog, embeddings, top_k=None)

def write_recommendations_step(recommendations):
    workflow_outputs["recommendations"] = recommendations

workflow = FullBatchRebuildWorkflow(
    fetch_catalog=fetch_catalog_step,
    enrich_catalog=partial(
        enrich_batch,
        llm=ENRICHMENT_LLM,
        prompt_version=PROMPT_VERSION,
    ),
    preprocess_catalog=preprocess_catalog_step,
    generate_embeddings=generate_embeddings_step,
    compute_recommendations=compute_recommendations_step,
    write_recommendations=write_recommendations_step,
)

artifacts = workflow.run()

print("\n[DEMO] Workflow artifacts summary:")

print(f"- Started at: {artifacts.started_at}")
print(f"- Finished at: {artifacts.finished_at}")
print(f"- Duration: {artifacts.duration_seconds:.2f} seconds")
print(f"- Raw datasets fetched: {artifacts.raw_catalog_size}")
print(f"- Processed datasets: {artifacts.processed_catalog_size}")
print(f"- Recommendation lists produced: {artifacts.recommendation_count}")

recommendations, dataset_profiles_processed = workflow_outputs["recommendations"], workflow_outputs["dataset_profiles"] 

with open(DEMO_OUTPUT_DIR / "datafinder_profiles.json", "w", encoding="utf-8") as f:
    json.dump([asdict(profile) for profile in dataset_profiles_processed], f, ensure_ascii=False, indent=2)


with open(DEMO_OUTPUT_DIR / "datafinder_recommendations.json", "w", encoding="utf-8") as f:
    json.dump(recommendations, f, ensure_ascii=False, indent=2)


##### Run the full batch workflow without LLM enrichment
##### We use the same workflow orchestration, but skip Bedrock and build embeddings directly from the raw metadata fields.

In [ ]:
baseline_workflow_outputs: dict[str, object] = {
    "recommendations": None,
}


def skip_enrichment_step(catalog: list[DatasetProfile]) -> list[DatasetProfile]:
    return catalog

def baseline_preprocess_catalog_step(catalog: list[DatasetProfile]) -> list[DatasetProfile]:
    return catalog

def baseline_generate_embeddings_step(catalog: list[DatasetProfile]):
    embedding_texts = [build_raw_embedding_text(profile) for profile in catalog]
    return encode_texts(embedding_texts, model_name=EMBEDDING_MODEL)

def baseline_write_recommendations_step(recommendations):
    baseline_workflow_outputs["recommendations"] = recommendations

baseline_workflow = FullBatchRebuildWorkflow(
    fetch_catalog=fetch_catalog_step,
    enrich_catalog=skip_enrichment_step,
    preprocess_catalog=baseline_preprocess_catalog_step,
    generate_embeddings=baseline_generate_embeddings_step,
    compute_recommendations=compute_recommendations_step,
    write_recommendations=baseline_write_recommendations_step,
)

baseline_artifacts = baseline_workflow.run()
baseline_recommendations = baseline_workflow_outputs["recommendations"]

print("\n[DEMO] Baseline workflow artifacts summary:")

print(f"- Started at: {baseline_artifacts.started_at}")
print(f"- Finished at: {baseline_artifacts.finished_at}")
print(f"- Duration: {baseline_artifacts.duration_seconds:.2f} seconds")
print(f"- Raw datasets fetched: {baseline_artifacts.raw_catalog_size}")
print(f"- Processed datasets: {baseline_artifacts.processed_catalog_size}")
print(f"- Recommendation lists produced: {baseline_artifacts.recommendation_count}")

with open(DEMO_OUTPUT_DIR / "datafinder_recommendations_no_llm.json", "w", encoding="utf-8") as f:
    json.dump(baseline_recommendations, f, ensure_ascii=False, indent=2)

##### We evaluate the generated ranking lists against the Datafinder ground-truth links.

In [6]:
with open(DEMO_OUTPUT_DIR / "datafinder_recommendations.json", "r", encoding="utf-8") as f:
    recommendations = json.load(f)

with open(DEMO_OUTPUT_DIR / "datafinder_recommendations_no_llm.json", "r", encoding="utf-8") as f:
    baseline_recommendations = json.load(f)

results_llm = {}
for n in EVAL_CUTOFFS:
    predictions = {
        item["id"]: [rec["id"] for rec in item.get("recommendations", [])][:n]
        for item in recommendations
    }
    results_llm[f"Recall@{n}"] = recall_at_n(predictions, ground_truth_links, n=n)
    results_llm[f"TNDCG@{n}"] = tndcg_at_n(predictions, ground_truth_links, n=n)

results_no_llm = {}
for n in EVAL_CUTOFFS:
    predictions = {
        item["id"]: [rec["id"] for rec in item.get("recommendations", [])][:n]
        for item in baseline_recommendations
    }
    results_no_llm[f"Recall@{n}"] = recall_at_n(predictions, ground_truth_links, n=n)
    results_no_llm[f"TNDCG@{n}"] = tndcg_at_n(predictions, ground_truth_links, n=n)

results_df = pd.DataFrame(
    [results_llm, results_no_llm],
    index=["Claude + SPECTER", "SPECTER"],
)
results_df

,Recall@10,TNDCG@10,Recall@20,TNDCG@20,Recall@50,TNDCG@50
Claude + SPECTER,0.541215,0.759029,0.671412,0.737650,0.812937,0.711068
SPECTER,0.476519,0.764360,0.635722,0.743442,0.806670,0.713717


In [7]:
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

console.print(Panel("[bold cyan]Impact of LLM Enrichment on Dataset Representations[/bold cyan]", expand=False))

EXAMPLE_DATASET_ID = "3DFAW"
TOP_K = 5

available_example_ids = {item["id"] for item in recommendations}
if EXAMPLE_DATASET_ID not in available_example_ids:
    raise ValueError(f"Dataset '{EXAMPLE_DATASET_ID}' is not available in the recommendation outputs.")

example_ids = [EXAMPLE_DATASET_ID]

with open(DEMO_OUTPUT_DIR / "datafinder_profiles.json", "r", encoding="utf-8") as f:
    profiles_data = json.load(f)

dataset_profiles_processed = [DatasetProfile(**p) for p in profiles_data]
dataset_profiles_baseline = profiles

id_to_profile_llm = {p.id: p for p in dataset_profiles_processed}
id_to_profile_baseline = {p.id: p for p in dataset_profiles_baseline}

# pick a few datasets deterministically
# NUM_EXAMPLES = 2
# example_ids = sorted([item["id"] for item in recommendations])[:NUM_EXAMPLES]

for dataset_id in example_ids:
    profile_llm = id_to_profile_llm.get(dataset_id)
    title = profile_llm.title if profile_llm else ""

    console.print(Panel(f"[bold]{dataset_id}[/bold]\n[italic]{title}[/italic]", title="Dataset", expand=False))

    recs_llm = next(item for item in recommendations if item["id"] == dataset_id)["recommendations"]
    recs_base = next(item for item in baseline_recommendations if item["id"] == dataset_id)["recommendations"]

    relevant_ids = ground_truth_links.get(dataset_id)

    if relevant_ids:
        llm_predictions = {
            dataset_id: [rec["id"] for rec in recs_llm]
        }
        base_predictions = {
            dataset_id: [rec["id"] for rec in recs_base]
        }
        single_ground_truth = {dataset_id: relevant_ids}

        llm_recall = recall_at_n(llm_predictions, single_ground_truth, n=TOP_K)
        llm_tndcg = tndcg_at_n(llm_predictions, single_ground_truth, n=TOP_K)
        base_recall = recall_at_n(base_predictions, single_ground_truth, n=TOP_K)
        base_tndcg = tndcg_at_n(base_predictions, single_ground_truth, n=TOP_K)

        llm_title = (
            f"Claude + SPECTER (Recall@{TOP_K}={llm_recall:.2f}, "
            f"TNDCG@{TOP_K}={llm_tndcg:.2f})"
        )
        base_title = (
            f"SPECTER (Recall@{TOP_K}={base_recall:.2f}, "
            f"TNDCG@{TOP_K}={base_tndcg:.2f})"
        )
    else:
        llm_title = f"Claude + SPECTER"
        base_title = f"SPECTER"

    # Create tables
    table_llm = Table(title=llm_title, show_lines=True)
    table_llm.add_column("Rank")
    table_llm.add_column("Dataset ID")
    table_llm.add_column("Score")
    table_llm.add_column("Relevant?")
    table_llm.add_column("Title")
    table_llm.add_column("SPECTER Input (truncated)")

    table_base = Table(title=base_title, show_lines=True)
    table_base.add_column("Rank")
    table_base.add_column("Dataset ID")
    table_base.add_column("Score")
    table_base.add_column("Relevant?")
    table_base.add_column("Title")
    table_base.add_column("SPECTER Input (truncated)")

    llm_ids = {rec["id"] for rec in recs_llm[:TOP_K]}
    base_ids = {rec["id"] for rec in recs_base[:TOP_K]}
    overlap_ids = llm_ids & base_ids & relevant_ids if relevant_ids else set()

    for i, rec in enumerate(recs_llm[:TOP_K], start=1):
        rec_profile = id_to_profile_llm.get(rec["id"])
        rec_title = rec_profile.title if rec_profile else ""
        if rec_profile:
            full_input = build_embedding_text(rec_profile)
            specter_input = full_input[:120] + "..."
        else:
            specter_input = ""
        if not relevant_ids:
            is_relevant = "-"
        else:
            is_relevant = "✓" if rec["id"] in relevant_ids else "✗"
        table_llm.add_row(str(i), rec["id"], f"{rec['score']:.4f}", is_relevant, rec_title, specter_input)

    for i, rec in enumerate(recs_base[:TOP_K], start=1):
        rec_profile = id_to_profile_baseline.get(rec["id"])
        rec_title = rec_profile.title if rec_profile else ""
        if rec_profile:
            full_input = build_raw_embedding_text(rec_profile)
            specter_input = full_input[:120] + "..."
        else:
            specter_input = ""
        if not relevant_ids:
            is_relevant = "-"
        else:
            is_relevant = "✓" if rec["id"] in relevant_ids else "✗"
        table_base.add_row(str(i), rec["id"], f"{rec['score']:.4f}", is_relevant, rec_title, specter_input)

    console.print(table_llm)
    console.print(table_base)

╭─────────────────────────────────────────────────────╮
│ Impact of LLM Enrichment on Dataset Representations │
╰─────────────────────────────────────────────────────╯

╭───────────────────────── Dataset ─────────────────────────╮
│ 3DFAW                                                     │
│ The First 3D Face Alignment in the Wild (3DFAW) Challenge │
╰───────────────────────────────────────────────────────────╯

                                  Claude + SPECTER (Recall@5=0.50, TNDCG@5=1.00)                                   
┏━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Rank ┃ Dataset ID ┃ Score  ┃ Relevant? ┃ Title                              ┃ SPECTER Input (truncated)         ┃
┡━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │ AFLW       │ 0.9747 │ ✓         │ Annotated Facial Landmarks in the  │ Annotated Facial Landmarks in the │
│      │            │        │           │ Wild: A large-scale, real-world    │ Wild: A large-scale, real-world   │
│      │            │        │           │ database for facial landmark       │ database for facial landmark      │
│      │            │        │           │ localization                       │ localization. The Annotat...      │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 2    │ 300W       │ 0.9703 │ ✗         │ 300 Faces in-the-Wild Challenge:   │ 300 Faces in-the-Wild Challenge:  │
│      │            │        │           │ The First Facial Landmark          │ The First Facial Landmark         │
│      │            │        │           │ Localization Challenge             │ Localization Challenge. The 300   │
│      │            │        │           │                                    │ Faces in-the-Wild (300-W) dat...  │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 3    │ WFLW       │ 0.9686 │ ✗         │ Look at Boundary: A Boundary-Aware │ Look at Boundary: A               │
│      │            │        │           │ Face Alignment Algorithm           │ Boundary-Aware Face Alignment     │
│      │            │        │           │                                    │ Algorithm. The Wider Facial       │
│      │            │        │           │                                    │ Landmarks in the Wild (WFLW)      │
│      │            │        │           │                                    │ dataset is a ...                  │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 4    │ LFPW       │ 0.9683 │ ✗         │ Localizing Parts of Faces Using a  │ Localizing Parts of Faces Using a │
│      │            │        │           │ Consensus of Exemplars             │ Consensus of Exemplars. The       │
│      │            │        │           │                                    │ Labeled Faces Parts in-the-Wild   │
│      │            │        │           │                                    │ (LFPW) dataset contains 1,...     │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 5    │ AFW        │ 0.9640 │ ✗         │ Face detection, pose estimation,   │ Face detection, pose estimation,  │
│      │            │        │           │ and landmark localization in the   │ and landmark localization in the  │
│      │            │        │           │ wild                               │ wild. The Annotated Faces in the  │
│      │            │        │           │                                    │ Wild (AFW) dataset is...          │
└──────┴────────────┴────────┴───────────┴────────────────────────────────────┴───────────────────────────────────┘

                                       SPECTER (Recall@5=0.50, TNDCG@5=0.50)                                       
┏━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Rank ┃ Dataset ID ┃ Score  ┃ Relevant? ┃ Title                              ┃ SPECTER Input (truncated)         ┃
┡━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1    │ WFLW       │ 0.9420 │ ✗         │ Look at Boundary: A Boundary-Aware │ Look at Boundary: A               │
│      │            │        │           │ Face Alignment Algorithm           │ Boundary-Aware Face Alignment     │
│      │            │        │           │                                    │ Algorithm. The Wider Facial       │
│      │            │        │           │                                    │ Landmarks in the Wild or WFLW     │
│      │            │        │           │                                    │ database con...                   │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 2    │ LFPW       │ 0.9303 │ ✗         │ Localizing Parts of Faces Using a  │ Localizing Parts of Faces Using a │
│      │            │        │           │ Consensus of Exemplars             │ Consensus of Exemplars. The       │
│      │            │        │           │                                    │ Labeled Face Parts in-the-Wild    │
│      │            │        │           │                                    │ (LFPW) consists of 1,432 fa...    │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 3    │ AFLW       │ 0.9269 │ ✓         │ Annotated Facial Landmarks in the  │ Annotated Facial Landmarks in the │
│      │            │        │           │ Wild: A large-scale, real-world    │ Wild: A large-scale, real-world   │
│      │            │        │           │ database for facial landmark       │ database for facial landmark      │
│      │            │        │           │ localization                       │ localization. The Annotat...      │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 4    │ Chairs     │ 0.9269 │ ✗         │ Seeing 3D Chairs: Exemplar         │ Seeing 3D Chairs: Exemplar        │
│      │            │        │           │ Part-Based 2D-3D Alignment Using a │ Part-Based 2D-3D Alignment Using  │
│      │            │        │           │ Large Dataset of CAD Models        │ a Large Dataset of CAD Models.    │
│      │            │        │           │                                    │ The Chairs dataset contains r...  │
├──────┼────────────┼────────┼───────────┼────────────────────────────────────┼───────────────────────────────────┤
│ 5    │ LFW        │ 0.9223 │ ✗         │ Labeled faces in the wild: A       │ Labeled faces in the wild: A      │
│      │            │        │           │ database for studying face         │ database for studying face        │
│      │            │        │           │ recognition in unconstrained       │ recognition in unconstrained      │
│      │            │        │           │ environments                       │ environments. The LFW dataset     │
│      │            │        │           │                                    │ conta...                          │
└──────┴────────────┴────────┴───────────┴────────────────────────────────────┴───────────────────────────────────┘

In [8]:
if overlap_ids:
    console.print(Panel("[bold]Full SPECTER Inputs for Overlapping Relevant Datasets[/bold]", expand=False))

    for oid in overlap_ids:
        llm_profile = id_to_profile_llm.get(oid)
        base_profile = id_to_profile_baseline.get(oid)

        console.print(Panel(f"[bold]{oid}[/bold]", title="Dataset", expand=False))

        if llm_profile:
            llm_full = build_embedding_text(llm_profile)
            console.print(Panel(llm_full, title="Claude + SPECTER", expand=False))

        if base_profile:
            base_full = build_raw_embedding_text(base_profile)
            console.print(Panel(base_full, title="SPECTER", expand=False))

╭───────────────────────────────────────────────────────╮
│ Full SPECTER Inputs for Overlapping Relevant Datasets │
╰───────────────────────────────────────────────────────╯

╭─ Dataset ─╮
│ AFLW      │
╰───────────╯

╭─────────────────────────────────────────────── Claude + SPECTER ────────────────────────────────────────────────╮
│ Annotated Facial Landmarks in the Wild: A large-scale, real-world database for facial landmark localization.    │
│ The Annotated Facial Landmarks in the Wild (AFLW) is a large-scale benchmark dataset comprising approximately   │
│ 25,000 real-world face images sourced from Flickr, each annotated with up to 21 facial keypoints including      │
│ nose, eyes, and ears. The dataset captures substantial intra-class variation across pose, expression,           │
│ ethnicity, age, gender, and diverse imaging conditions, making it representative of unconstrained, in-the-wild  │
│ scenarios. Its primary value lies in providing richly labeled ground-truth data for training and evaluating     │
│ facial analysis algorithms. AFLW is intended for computer vision researchers and machine learning practitioners │
│ working on facial landmark localization, face alignment, head pose estimation, and unsupervised keypoint        │
│ detection. Specific use cases include benchmarking landmark regression models, developing robust pose-invariant │
│ face alignment pipelines, and supporting low-light image enhancement research where facial structure recovery   │
│ is required.                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── SPECTER ────────────────────────────────────────────────────╮
│ Annotated Facial Landmarks in the Wild: A large-scale, real-world database for facial landmark localization.    │
│ The Annotated Facial Landmarks in the Wild (AFLW) is a large-scale collection of annotated face images gathered │
│ from Flickr, exhibiting a large variety in appearance (e.g., pose, expression, ethnicity, age, gender) as well  │
│ as general imaging and environmental conditions. In total about 25K faces are annotated with up to 21 landmarks │
│ per image. Source: Nose, Eyes and Ears: Head Pose Estimation by Locating Facial Keypoints. Face Alignment,      │
│ Facial Landmark Detection, Low-Light Image Enhancement, Head Pose Estimation, Unsupervised Facial Landmark      │
│ Detection                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯